# 🌊 VaayuChakshu — Flood Monitoring with SAR-to-Optical GAN
**Author: Karthik Raveendran**

This notebook trains the Conditional GAN model for SAR-to-Optical image translation and flood water segmentation on Google Colab T4 GPU.

### Steps:
1. ✅ Check GPU
2. ✅ Mount Google Drive
3. ✅ Upload project ZIP from Drive
4. ✅ Install dependencies
5. ✅ Run training
6. ✅ Run inference

## Step 1 — Check GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
!nvidia-smi

## Step 2 — Mount Google Drive
Upload `VaayuChakshu.zip` to your Google Drive root folder first, then run this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

## Step 3 — Extract Project from Google Drive

In [ ]:
import os

# Copy zip from Drive to Colab
!cp /content/drive/MyDrive/VaayuChakshu.zip /content/

# Extract
!unzip -q /content/VaayuChakshu.zip -d /content/

# Set working directory
os.chdir('/content/VaayuChakshu')
print('Working directory:', os.getcwd())
!ls

## Step 4 — Install Dependencies
On Colab (Linux), all packages install cleanly including CUDA support.

In [ ]:
# Install all requirements (Linux version — no need to comment out nvidia-* packages)
# We skip torch/torchvision as Colab already has them with CUDA
!pip install -q \
    pytorch-lightning==2.5.2 \
    lightning==2.5.2 \
    wandb==0.20.1 \
    rasterio==1.4.3 \
    albumentations==2.0.8 \
    torchmetrics==1.7.3 \
    onnx==1.18.0 \
    onnxruntime-gpu==1.22.0 \
    python-dotenv==1.1.0 \
    scikit-image==0.25.2 \
    tifffile==2025.6.11 \
    awscli==1.40.40

print('All dependencies installed!')

In [ ]:
# Verify GPU is still available after installs
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

## Step 5 — Configure for Colab
Update config to use Colab-optimised settings (more workers, larger batch).

In [ ]:
import yaml, os

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Colab-optimised settings
config['num_workers'] = 2          # Colab supports multiprocessing
config['persistent_workers'] = True
config['pin_memory'] = True
config['batch_size'] = 4           # T4 has 16GB VRAM, conservative start
config['accelerator'] = 'gpu'
config['precision'] = '16-mixed'   # Use Tensor Cores on T4

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config updated for Colab T4:')
for k, v in config.items():
    print(f'  {k}: {v}')

## Step 6 — Download Training Data from AWS S3
*(Skip this if data is already in the zip)*

In [ ]:
import os

# Check if data already exists in the zip
manifest = 'data/processed/data_manifest.csv'
if os.path.exists(manifest):
    import pandas as pd
    df = pd.read_csv(manifest)
    print(f'Data manifest found! {len(df)} total samples')
    print(df['split'].value_counts())
else:
    print('No manifest found. Downloading data from AWS S3...')
    !python scripts/fetch_data.py
    !python scripts/prepare_dataset.py

## Step 7 — 🚀 Start Training!
Training runs for 200 epochs. On T4 GPU with 23 samples, each epoch takes ~5-10 seconds.
**Total estimated time: ~30-40 minutes** (much faster than local laptop!)

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'  # No internet needed for logging

# Add src to path
import sys
sys.path.insert(0, 'src')

# Set Tensor Core precision for T4
import torch
torch.set_float32_matmul_precision('medium')

!python -m src.train --config config.yaml

## Step 8 — Save Checkpoints to Google Drive
Colab sessions reset after ~12 hours — save your trained model to Drive!

In [ ]:
import shutil, os

drive_save_path = '/content/drive/MyDrive/VaayuChakshu_checkpoints'
os.makedirs(drive_save_path, exist_ok=True)

if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', drive_save_path, dirs_exist_ok=True)
    print(f'Checkpoints saved to Google Drive: {drive_save_path}')
    !ls -lh /content/drive/MyDrive/VaayuChakshu_checkpoints/
else:
    print('No checkpoints folder found yet. Training may still be running.')

## Step 9 — Run Inference
After training, test the model on a sample image.

In [ ]:
import glob

# Find the best checkpoint
ckpts = glob.glob('checkpoints/**/*.ckpt', recursive=True)
print('Available checkpoints:')
for c in ckpts:
    print(' ', c)

# Find a sample SAR image to test on
sar_samples = glob.glob('data/raw/**/VV.tif', recursive=True)
print(f'\nFound {len(sar_samples)} SAR samples for inference')
if sar_samples:
    print('First sample:', sar_samples[0])

In [ ]:
# Run inference if checkpoint exists
import glob, os
ckpts = glob.glob('checkpoints/**/last.ckpt', recursive=True)

if ckpts:
    checkpoint = ckpts[0]
    print(f'Running inference with: {checkpoint}')
    !python inference.py --checkpoint {checkpoint} --output outputs/
else:
    print('No checkpoint found. Please complete training first (Step 7).')

## Step 10 — Visualise Results

In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import rasterio

output_files = glob.glob('outputs/**/*.tif', recursive=True) + glob.glob('outputs/**/*.png', recursive=True)

if output_files:
    fig, axes = plt.subplots(1, min(3, len(output_files)), figsize=(15, 5))
    if len(output_files) == 1:
        axes = [axes]
    for ax, f in zip(axes, output_files[:3]):
        with rasterio.open(f) as src:
            img = src.read()
            img = np.moveaxis(img[:3], 0, -1)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img)
        ax.set_title(f.split('/')[-1])
        ax.axis('off')
    plt.suptitle('VaayuChakshu — SAR to Optical Translation Results', fontsize=14)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/VaayuChakshu_results.png', dpi=150)
    plt.show()
    print('Results saved to Google Drive!')
else:
    print('No output files found. Run inference first.')